# SPFI ×4 — trained width comparison C8 vs C16 on the validation split (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/main/notebooks/02_width_eval_val_colab.ipynb)

Compares two SPFI models that differ **only in width** (`embed_dim` 8 vs 16), trained with
`train_colab.ipynb` under the same budget (same `MAX_ITERS`, seeds, data and settings).

* **Quality:** PSNR / SSIM per image on the **validation split** (the 20% of EMPS `train.csv` images the
  trainer held out, reproduced with `algo.train.split_files`), plus bicubic as a reference.
  **The test split is never downloaded** — it stays locked for the final evaluation.
* **What the eye sees:** PSNR rewards blur (averaging is the safest guess), so it can favour bicubic over
  a model that restores detail. Two extra views address that: **LPIPS** (a learned perceptual distance,
  lower = closer to HR as people judge it) and **edge metrics** (how sharp the edges are relative to HR,
  and how far they are from HR's edges), plus a **visual comparison** with zoomed crops.
* **Cost:** parameters, FLOPs, inference latency and peak memory (this GPU), and training time / memory
  from the `train_info_*.json` files written by `train_colab.ipynb`.
* **Paired analysis:** both models see the same images, so the per-image difference C16 − C8 is reported
  with a bootstrap 95% confidence interval.

> **Status: exploratory.** One training seed per width, a fixed short budget, validation split only.
> These numbers are a first look, not the approved comparison (that waits for approved checkpoints, #156).

**Before you start:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.

## Settings
Point `CHECKPOINTS` at the two **final** files from `train_colab.ipynb` (both trained for exactly `MAX_ITERS`).
`RUN_SUFFIX` picks the run: `""` for the 5000-iteration models, `"_it20000"` for the 20000-iteration ones.
Leave a path empty to upload the file from your computer. `TRAIN_INFO` is optional (adds training cost).

In [ ]:
import os
from pathlib import Path

# ---- Code ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "main"

# ---- Data: public EMPS release (same as train_colab.ipynb); only train.csv images are extracted ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"
SPLIT_SEED = 0          # must equal SPLIT_SEED used for training -> same validation images

# ---- The two models (label -> checkpoint path). Empty string = upload. ----
RUN_SUFFIX = ""         # "" = the 5000-iteration runs (…_e8_s0.pt); "_it20000" = the 20000-iteration runs
DRIVE = "/content/drive/MyDrive/spfi_checkpoints"
CHECKPOINTS = {
    "C8":  f"{DRIVE}/spfi_x4_final_e8_s0{RUN_SUFFIX}.pt",
    "C16": f"{DRIVE}/spfi_x4_final_e16_s0{RUN_SUFFIX}.pt",
}
EXPECTED_EMBED_DIM = {"C8": 8, "C16": 16}
TRAIN_INFO = {          # optional; missing files are skipped
    "C8":  f"{DRIVE}/train_info_e8_s0{RUN_SUFFIX}.json",
    "C16": f"{DRIVE}/train_info_e16_s0{RUN_SUFFIX}.json",
}

# ---- Evaluation (same tiling as inference_colab.ipynb) ----
TILE = 64            # LR tile size (= training LR patch size)
OVERLAP = 16         # LR pixels shared by neighbouring tiles
BATCH_TILES = 16     # tiles per forward pass
SHAVE = 4            # HR border pixels ignored by PSNR / SSIM
MAX_VAL_IMAGES = None    # None = whole validation split; a number = quick check on the first N
BOOTSTRAP = 10_000   # resamples for the confidence interval of the paired difference
TIMING_RUNS = 50     # forward passes timed per model (after warm-up)

IN_COLAB = "google.colab" in __import__("sys").modules
WORK_DIR = Path("/content") if IN_COLAB else Path(os.environ.get("SR_WORK_DIR", Path.cwd() / "_width_eval"))
CODE_DIR = WORK_DIR / "SR_amat_3"
DATA_DIR = WORK_DIR / "emps"
OUTPUT_DIR = WORK_DIR / "output"
# Local testing only: environment variables override the values above.
CODE_REPO_URL = os.environ.get("SR_REPO_URL") or CODE_REPO_URL
CODE_BRANCH = os.environ.get("SR_REF") or CODE_BRANCH
if os.environ.get("SR_CKPT_C8"):
    CHECKPOINTS = {"C8": os.environ["SR_CKPT_C8"], "C16": os.environ["SR_CKPT_C16"]}
    TRAIN_INFO = {"C8": os.environ.get("SR_INFO_C8", ""), "C16": os.environ.get("SR_INFO_C16", "")}
if os.environ.get("SR_MAX_VAL_IMAGES"):
    MAX_VAL_IMAGES = int(os.environ["SR_MAX_VAL_IMAGES"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Code and dependencies

In [ ]:
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pytorch_wavelets>=1.3", "PyWavelets>=1.6", "setuptools<81", "lpips>=0.1.4"], check=True)

In [ ]:
import hashlib
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout.strip()
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def download_emps():
    """Download the pinned public EMPS archive once (no credentials needed) and check its SHA-256."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    return archive


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_train():
    """Extract train.csv, test.csv and only the train.csv images into DATA_DIR/train/."""
    dest = DATA_DIR / "train"
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_emps()) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / "train.csv").read_text())
        for image_id in ids:
            out = dest / "images" / f"{image_id}.png"
            if not out.exists():
                out.write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", CODE_REPO_URL, str(CODE_DIR))
git("fetch", "--quiet", "origin", CODE_BRANCH, cwd=CODE_DIR)
git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
COMMIT = git("rev-parse", "HEAD", cwd=CODE_DIR)
print("code:", git("log", "-1", "--format=%h %s", cwd=CODE_DIR))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

## Validation images
The trainer splits the `train.csv` images 80/20 with `SPLIT_SEED`; `split_files` reproduces that split.
The test images are not extracted, and the check below makes sure none is used.

In [ ]:
from algo.train import split_files

EMPS_DIR, train_ids = extract_emps_train()
test_ids = set(read_split((EMPS_DIR / "test.csv").read_text()))
on_disk = sorted(p.stem for p in (EMPS_DIR / "images").glob("*.png"))
assert on_disk == sorted(train_ids), "images/ must hold exactly the train split"
assert not set(on_disk) & test_ids, "test images must never be used here"

train_files, val_ids = split_files(on_disk, SPLIT_SEED)
if MAX_VAL_IMAGES is not None:
    val_ids = val_ids[:MAX_VAL_IMAGES]
print(f"{len(on_disk)} train.csv images -> {len(train_files)} used for training / "
      f"{len(val_ids)} validation images evaluated here (split seed {SPLIT_SEED}); test split untouched")

## Load both models
Each checkpoint is loaded the way the local app loads it. The width must match the label, and both must
have the same number of training iterations — otherwise the comparison is not fixed-budget.

In [ ]:
import json
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import torch

from algo.model import build_spfi

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    TIMING_RUNS = min(TIMING_RUNS, 3)   # CPU is far slower; keep the run short
    print("WARNING: no GPU — quality numbers are still valid, timing is CPU timing (only a few runs).")


def resolve(path, label):
    if path and path.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if path and Path(path).exists():
        return Path(path)
    if IN_COLAB:
        from google.colab import files
        print(f"Upload the {label} file" + (f" ({Path(path).name})" if path else ""))
        name, data = next(iter(files.upload().items()))
        (WORK_DIR / name).write_bytes(data)
        return WORK_DIR / name
    return None


MODELS, META = {}, {}
for label, path in CHECKPOINTS.items():
    p = resolve(path, f"{label} checkpoint")
    assert p is not None, f"{label}: checkpoint not found: {path}"
    ckpt = torch.load(p, map_location=device, weights_only=True)
    hp = ckpt["hparams"]
    model = build_spfi(scale=hp["scale"], embed_dim=hp["embed_dim"])
    model.load_state_dict(ckpt["model_state"])
    MODELS[label] = model.to(device).eval()
    META[label] = {
        "file": p.name,
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        "embed_dim": hp["embed_dim"],
        "scale": hp["scale"],
        "iteration": ckpt.get("iteration"),
        "params": sum(q.numel() for q in model.parameters()),
    }
    print(f"{label}: {p.name}  embed_dim={hp['embed_dim']}  iteration={ckpt.get('iteration')}  "
          f"params={META[label]['params']:,}  sha256={META[label]['sha256'][:12]}…")

SCALE = META["C8"]["scale"]
for label, m in META.items():
    assert m["embed_dim"] == EXPECTED_EMBED_DIM[label], f"{label} has embed_dim {m['embed_dim']}"
    assert m["scale"] == SCALE == 4
SAME_BUDGET = META["C8"]["iteration"] == META["C16"]["iteration"]
print("same training iterations:", SAME_BUDGET,
      "" if SAME_BUDGET else "  <-- NOT a fixed-budget comparison; interpret with care")

TRAIN_INFOS = {}
for label, path in TRAIN_INFO.items():
    if path and path.startswith("/content/drive") and IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    if path and Path(path).exists():
        TRAIN_INFOS[label] = json.loads(Path(path).read_text())
print("training info:", ", ".join(TRAIN_INFOS) or "none (training cost columns will be empty)")

## Quality on the validation split
Same procedure as `inference_colab.ipynb`: degrade each HR image with the repo's `degrade`, super-resolve
with overlapping 64×64 tiles, compare to HR with `psnr` / `ssim_loss` from `algo.train`, ignoring a
`SHAVE`-pixel border. Bicubic upscaling of the same LR image is the reference.

Per image and per method it also records:
* **LPIPS** (AlexNet, grayscale repeated to 3 channels): perceptual distance to HR, **lower is better**.
* **Edge sharpness** = mean Sobel gradient magnitude of the output / of HR. 1.0 = as sharp as HR,
  below 1 = blurrier, above 1 = harsher (over-sharpened or noisy).
* **Edge error** = mean |gradient(output) − gradient(HR)|, **lower is better**: are the edges in the right place
  with the right strength (relevant for particle boundaries)?

In [ ]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm

from algo.image_loading import load_image
from algo.preprocessing import normalize_image
from algo.train import degrade, psnr, ssim_loss


def tile_starts(n):
    """Start offsets of TILE-sized windows covering 0..n, OVERLAP apart, the last one flush with the end."""
    if n <= TILE:
        return [0]
    starts = list(range(0, n - TILE, TILE - OVERLAP))
    return starts + [n - TILE]


@torch.no_grad()
def super_resolve(model, lr):
    """[1, 1, h, w] LR -> [1, 1, h*SCALE, w*SCALE] SR, tiled exactly as in inference_colab.ipynb."""
    _, _, h, w = lr.shape
    lr = F.pad(lr, (0, max(0, TILE - w), 0, max(0, TILE - h)), mode="replicate")
    H, W = lr.shape[-2:]
    coords = [(y, x) for y in tile_starts(H) for x in tile_starts(W)]
    out = torch.zeros(1, 1, H * SCALE, W * SCALE, device=lr.device)
    hits = torch.zeros_like(out)
    t = TILE * SCALE
    for i in range(0, len(coords), BATCH_TILES):
        chunk = coords[i:i + BATCH_TILES]
        sr = model(torch.cat([lr[..., y:y + TILE, x:x + TILE] for y, x in chunk]))
        for (y, x), tile in zip(chunk, sr):
            out[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += tile
            hits[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += 1
    return (out / hits)[..., :h * SCALE, :w * SCALE]


def shave(img):
    return img[..., SHAVE:-SHAVE, SHAVE:-SHAVE] if SHAVE else img


try:
    import lpips
    LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
except Exception as e:   # not installed, or the AlexNet weights cannot be downloaded
    LPIPS_NET = None
    print(f"LPIPS unavailable ({type(e).__name__}: {e}) — LPIPS columns will be empty")

_SOBEL = torch.tensor([[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]])


def grad_mag(img):
    """Sobel gradient magnitude of [1, 1, H, W], without the 1-pixel padded border."""
    kx = _SOBEL.view(1, 1, 3, 3).to(img)
    ky = _SOBEL.t().reshape(1, 1, 3, 3).to(img)
    g = torch.sqrt(F.conv2d(img, kx, padding=1) ** 2 + F.conv2d(img, ky, padding=1) ** 2 + 1e-12)
    return g[..., 1:-1, 1:-1]


@torch.no_grad()
def lpips_dist(out, ref):
    if LPIPS_NET is None:
        return float("nan")
    to3 = lambda t: (t.to(device) * 2 - 1).repeat(1, 3, 1, 1)   # [0,1] gray -> [-1,1] RGB
    return LPIPS_NET(to3(out), to3(ref)).item()


def score(out, hr):
    """All metrics of one output vs HR (both [1, 1, H, W] in [0, 1]), on the shaved region."""
    o, h = shave(out), shave(hr)
    go, gh = grad_mag(o), grad_mag(h)
    return {"psnr": psnr(o, h), "ssim": 1.0 - ssim_loss(o, h).item(), "lpips": lpips_dist(o, h),
            "sharpness": (go.mean() / gh.mean()).item(), "edge_err": (go - gh).abs().mean().item()}


def process(image_id):
    """HR, bicubic and each model's SR for one image (all [1, 1, H, W] on CPU, in [0, 1])."""
    hr = to_tensor(normalize_image(load_image(EMPS_DIR / "images" / f"{image_id}.png")))   # [1, H, W]
    _, H, W = hr.shape
    hr = hr[:, :H - H % SCALE, :W - W % SCALE]
    lr = degrade(hr, filename=image_id, scale=SCALE)
    outs = {"bicubic": F.interpolate(lr.unsqueeze(0), scale_factor=SCALE, mode="bicubic",
                                     align_corners=False).clamp(0, 1)}
    for label, model in MODELS.items():
        outs[label] = super_resolve(model, lr.unsqueeze(0).to(device)).clamp(0, 1).cpu()
    return hr.unsqueeze(0), outs


METHODS = ["bicubic", *MODELS]
METRICS = ["psnr", "ssim", "lpips", "sharpness", "edge_err"]
rows = []
for image_id in tqdm(val_ids):
    hr, outs = process(image_id)
    row = {"image": image_id, "hr_size": f"{hr.shape[-1]}x{hr.shape[-2]}"}
    for method, out in outs.items():
        row.update({f"{method}_{k}": v for k, v in score(out, hr).items()})
    rows.append(row)

results = pd.DataFrame(rows)
for k in METRICS:
    results[f"delta_{k}"] = results[f"C16_{k}"] - results[f"C8_{k}"]
results.to_csv(OUTPUT_DIR / "val_results.csv", index=False)
results.head()

## Paired difference C16 − C8
Both models were evaluated on the same images, so the per-image difference removes image difficulty.
The 95% interval comes from resampling images with replacement (bootstrap). If it includes 0, this
validation set cannot tell the two widths apart at this budget.

In [ ]:
rng = np.random.default_rng(0)


def paired_summary(delta, higher_is_better=True):
    delta = np.asarray(delta, dtype=float)
    delta = delta[~np.isnan(delta)]
    if len(delta) == 0:
        return {"mean": np.nan, "ci95_low": np.nan, "ci95_high": np.nan, "C16_better": 0, "C8_better": 0, "n": 0}
    boots = rng.choice(delta, size=(BOOTSTRAP, len(delta)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    better = delta > 0 if higher_is_better else delta < 0
    worse = delta < 0 if higher_is_better else delta > 0
    return {"mean": delta.mean(), "ci95_low": lo, "ci95_high": hi,
            "C16_better": int(better.sum()), "C8_better": int(worse.sum()), "n": len(delta)}


PAIRED = {"PSNR (dB)": paired_summary(results["delta_psnr"]),
          "SSIM": paired_summary(results["delta_ssim"]),
          "LPIPS (lower = better)": paired_summary(results["delta_lpips"], higher_is_better=False),
          "edge error (lower = better)": paired_summary(results["delta_edge_err"], higher_is_better=False)}
paired_table = pd.DataFrame(PAIRED).T
paired_table["C16 better on"] = [f"{int(r['C16_better'])}/{int(r['n'])} images" for _, r in paired_table.iterrows()]
display(paired_table[["mean", "ci95_low", "ci95_high", "C16 better on"]].astype({"mean": float}).round(4))

## Cost
* **Params / FLOPs:** per 64×64 LR tile (256×256 output).
* **Latency / memory:** median time of one forward pass on a batch of `BATCH_TILES` tiles after warm-up,
  and peak GPU memory of that pass, on **this** GPU. Re-measure on the same GPU before comparing to other runs.
* **Training:** from `train_info_*.json` (time, seconds per iteration, peak memory), if provided.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


@torch.no_grad()
def measure(model):
    x = torch.rand(BATCH_TILES, 1, TILE, TILE, device=device)
    with FlopCounterMode(display=False) as counter:
        model(x[:1])
    gflops = counter.get_total_flops() / 1e9
    for _ in range(5):
        model(x)
    times = []
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    for _ in range(TIMING_RUNS):
        t0 = time.perf_counter()
        model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    peak = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else None
    return {"gflops_per_tile": gflops, "latency_ms_per_batch": 1000 * float(np.median(times)), "peak_infer_mem_mb": peak}


COST = {}
for label, model in MODELS.items():
    info = TRAIN_INFOS.get(label, {})
    COST[label] = {
        "params": META[label]["params"],
        **measure(model),
        "train_minutes": info.get("train_seconds", np.nan) / 60 if info else np.nan,
        "train_s_per_iter": info.get("seconds_per_iter", np.nan),
        "train_peak_mem_mb": info.get("peak_gpu_memory_mb", np.nan),
        "train_gpu": info.get("gpu"),
    }
cost_table = pd.DataFrame(COST).T
cost_table.loc["C16 / C8"] = [
    (cost_table.loc["C16", c] / cost_table.loc["C8", c]) if isinstance(cost_table.loc["C8", c], (int, float, np.floating)) and cost_table.loc["C8", c] else None
    for c in cost_table.columns]
print("timing device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU",
      f"| batch {BATCH_TILES}×{TILE}×{TILE}")
display(cost_table)

## Summary table and plots

In [ ]:
import matplotlib.pyplot as plt

summary = pd.DataFrame({
    label: {
        "embed_dim": META[label]["embed_dim"] if label in META else "—",
        "iterations": META[label]["iteration"] if label in META else "—",
        "params": f"{META[label]['params']:,}" if label in META else "—",
        "mean PSNR (dB)": round(results[f"{key}_psnr"].mean(), 3),
        "mean SSIM": round(results[f"{key}_ssim"].mean(), 4),
        "mean LPIPS (lower = better)": round(results[f"{key}_lpips"].mean(), 4),
        "edge sharpness (1 = HR)": round(results[f"{key}_sharpness"].mean(), 3),
        "edge error (lower = better)": round(results[f"{key}_edge_err"].mean(), 4),
        "PSNR gain vs bicubic (dB)": round((results[f"{key}_psnr"] - results["bicubic_psnr"]).mean(), 3),
        "beats bicubic PSNR on": "—" if key == "bicubic" else f"{int((results[f'{key}_psnr'] > results['bicubic_psnr']).sum())}/{len(results)}",
        "beats bicubic LPIPS on": "—" if key == "bicubic" else f"{int((results[f'{key}_lpips'] < results['bicubic_lpips']).sum())}/{len(results)}",
    }
    for label, key in (("bicubic", "bicubic"), ("C8", "C8"), ("C16", "C16"))
}).T
display(summary)

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
labels = ["bicubic", "C8", "C16"]
colors = ["#9AA0A6", "#4C72B0", "#DD8452"]
for ax, metric, title in ((axes[0], "psnr", "PSNR (dB) ↑"), (axes[1], "ssim", "SSIM ↑"),
                          (axes[2], "lpips", "LPIPS ↓ (perceptual)")):
    means = [results[f"{k}_{metric}"].mean() for k in labels]
    if np.isnan(means).any():
        ax.set_title(f"{title}: unavailable")
        ax.axis("off")
        continue
    bars = ax.bar(labels, means, color=colors, width=0.55)
    ax.bar_label(bars, labels=[f"{m:.3f}" if metric == "psnr" else f"{m:.4f}" for m in means], padding=3)
    span = max(means) - min(means) or abs(max(means)) * 0.05 or 1.0
    ax.set_ylim(min(means) - 0.8 * span, max(means) + 0.5 * span)
    ax.set_title(f"mean {title}, {len(results)} val images")
    ax.spines[["top", "right"]].set_visible(False)

ax = axes[3]
ax.scatter(results["C8_psnr"], results["C16_psnr"], s=14, alpha=0.7, color="#DD8452")
lims = [min(results[["C8_psnr", "C16_psnr"]].min()), max(results[["C8_psnr", "C16_psnr"]].max())]
ax.plot(lims, lims, color="#5F6368", lw=1, ls="--")
ax.set_xlabel("C8 PSNR (dB)")
ax.set_ylabel("C16 PSNR (dB)")
ax.set_title("per image (above the line = C16 better)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "width_comparison.png", dpi=130)
plt.show()

## Training curves
Validation PSNR / loss (every 250 iterations, on the trainer's 64 fixed validation patches) and the
training loss (every 10 iterations, smoothed) of each model, read from the Drive folder of the checkpoints:
* `spfi_run_<tag>_<time>.zip`: the run log `train_colab.ipynb` exports at the end of a run;
* `metrics_<tag>_<time>.jsonl`: the same log, copied to Drive during training (survives a disconnect);
* `console_<tag>_session<N>.txt`: notebook output pasted into a text file (validation lines only), for
  sessions whose log was lost (e.g. `console_e8_s0_it20000_session1.txt`).

All sources of a model are joined by iteration (a later file wins on overlaps). The dashed line is bicubic on
the same validation patches. These are training-time patch metrics; the full-image numbers above are the
comparison itself.

In [ ]:
import io
import re
import zipfile

LOG_DIR = Path(CHECKPOINTS["C8"]).parent
VAL_LINE = re.compile(r"\[iter\s+(\d+)\]\s+loss=([\d.]+)\s+val_loss=([\d.]+)\s+val_psnr=([\d.]+)")


def read_jsonl(text, source):
    train, val = [], []
    for line in text.splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if r.get("type") == "train":
            train.append({"step": r["step"], "loss": r["loss"], "source": source})
        elif r.get("type") == "val":
            val.append({"step": r["step"], "val_loss": r["val_loss"], "val_psnr": r["val_psnr"],
                        "val_ssim": r.get("val_ssim", np.nan), "source": source})
    return train, val


def load_curves(tag):
    """(train, val, bicubic_psnr, sources) for one run tag, joined over every log file found."""
    if not LOG_DIR.exists():
        return pd.DataFrame(), pd.DataFrame(), None, []
    train, val, bicubic, sources = [], [], None, []
    files = sorted((f for f in LOG_DIR.iterdir()
                    if re.fullmatch(rf"spfi_run_{re.escape(tag)}_\d{{8}}-\d{{6}}\.zip", f.name)
                    or re.fullmatch(rf"metrics_{re.escape(tag)}_\d{{8}}-\d{{6}}\.jsonl", f.name)
                    or re.fullmatch(rf"console_{re.escape(tag)}(_session\w*)?\.txt", f.name)),
                   key=lambda f: f.stat().st_mtime)
    for f in files:
        sources.append(f.name)
        if f.suffix == ".zip":
            with zipfile.ZipFile(f) as z:
                names = z.namelist()
                if "metrics.jsonl" in names:
                    t, v = read_jsonl(z.read("metrics.jsonl").decode(), f.name)
                    train += t
                    val += v
                if "meta.json" in names:
                    bicubic = json.loads(z.read("meta.json")).get("bicubic_psnr", bicubic)
        elif f.suffix == ".jsonl":
            t, v = read_jsonl(f.read_text(), f.name)
            train += t
            val += v
            meta = LOG_DIR / f.name.replace("metrics_", "meta_").replace(".jsonl", ".json")
            if meta.exists():
                bicubic = json.loads(meta.read_text()).get("bicubic_psnr", bicubic)
        else:
            for m in VAL_LINE.finditer(f.read_text()):
                val.append({"step": int(m[1]), "val_loss": float(m[3]), "val_psnr": float(m[4]),
                            "val_ssim": np.nan, "source": f.name})
    dedupe = lambda rows: (pd.DataFrame(rows).drop_duplicates("step", keep="last").sort_values("step")
                           if rows else pd.DataFrame())
    return dedupe(train), dedupe(val), bicubic, sources


CURVES = {}
for label in MODELS:
    tag = f"e{EXPECTED_EMBED_DIM[label]}_s0{RUN_SUFFIX}"
    CURVES[label] = load_curves(tag)
    tr, va, bic, src = CURVES[label]
    print(f"{label} ({tag}): {len(va)} validation points, {len(tr)} train points from {len(src)} file(s): "
          + (", ".join(src) or "none found"))

if not any(len(v[1]) for v in CURVES.values()):
    print("No training logs found next to the checkpoints — skipping the curves.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
    colors = {"C8": "#4C72B0", "C16": "#DD8452"}
    bicubic_ref = next((c[2] for c in CURVES.values() if c[2] is not None), None)
    for label, (tr, va, bic, _) in CURVES.items():
        if len(va):
            axes[0].plot(va["step"], va["val_psnr"], color=colors[label], label=label, lw=1.4)
            axes[1].plot(va["step"], va["val_loss"], color=colors[label], label=label, lw=1.4)
        if len(tr):
            smooth = tr["loss"].rolling(50, min_periods=1).mean()   # 50 points = 500 iterations
            axes[2].plot(tr["step"], smooth, color=colors[label], label=label, lw=1.2)
    if bicubic_ref is not None:
        axes[0].axhline(bicubic_ref, color="#5F6368", ls="--", lw=1, label=f"bicubic ({bicubic_ref:.2f} dB)")
    for ax, title in zip(axes, ("validation PSNR (dB) ↑", "validation loss ↓", "training loss ↓ (moving average, 500 it)")):
        ax.set_title(title)
        ax.set_xlabel("iteration")
        ax.legend(frameon=False)
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=120)
    plt.show()

    rows = []
    for label, (tr, va, bic, _) in CURVES.items():
        if not len(va):
            continue
        above = va[va["val_psnr"] > bicubic_ref] if bicubic_ref is not None else va.iloc[0:0]
        last = va.iloc[-1]
        tail = va[va["step"] > last["step"] - 2500]["val_psnr"]   # last 10 validations
        rows.append({"model": label, "last iteration": int(last["step"]),
                     "last val PSNR": round(last["val_psnr"], 2), "best val PSNR": round(va["val_psnr"].max(), 2),
                     "first iteration above bicubic": int(above["step"].iloc[0]) if len(above) else "—",
                     "PSNR gain over last 2500 it": round(tail.iloc[-1] - tail.iloc[0], 2) if len(tail) > 1 else "—"})
    display(pd.DataFrame(rows).set_index("model"))

## Visual comparison
PSNR and LPIPS are summaries; this is what the outputs actually look like. Rows are chosen automatically
to cover different cases (largest model win and loss vs bicubic, largest C16 − C8 gain, the worst C8 SSIM
failure, a typical image). Each row: HR with the zoom box, then the same zoomed crop of HR, bicubic, C8
and C16, all with the same gray range (0–1) and nearest-neighbour display (no extra smoothing).
The crop is the HR region with the most edge detail, skipping burned-in labels and scale bars.
Set `VISUAL_IMAGES` to choose images yourself.

In [ ]:
from matplotlib.patches import Rectangle

VISUAL_IMAGES = None    # e.g. ["af674cfe7e", "9dc60257e5"]; None = automatic picks
CROP = 128              # HR pixels


def pick_cases(r):
    gain = r["C16_psnr"] - r["bicubic_psnr"]
    c8_ssim_gap = r["bicubic_ssim"] - r["C8_ssim"]
    typical = (r["delta_psnr"] - r["delta_psnr"].median()).abs()
    cases = {
        "C16 beats bicubic most (PSNR)": r.loc[gain.idxmax(), "image"],
        "C16 loses to bicubic most (PSNR)": r.loc[gain.idxmin(), "image"],
        "largest C16 − C8 gain": r.loc[r["delta_psnr"].idxmax(), "image"],
        "worst C8 SSIM vs bicubic": r.loc[c8_ssim_gap.idxmax(), "image"],
        "typical C16 − C8": r.loc[typical.idxmin(), "image"],
    }
    if not r["C16_lpips"].isna().all():
        cases["C16 beats bicubic most (LPIPS)"] = r.loc[(r["bicubic_lpips"] - r["C16_lpips"]).idxmax(), "image"]
    seen, out = set(), {}
    for why, img in cases.items():
        if img not in seen:
            seen.add(img)
            out[why] = img
    return out


def detail_crop(hr):
    """(y, x) of the CROP×CROP HR window with the most edge energy, avoiding burned-in labels / scale bars
    (EMPS images contain text; it is pure black/white, so windows with saturated pixels are skipped)."""
    g = grad_mag(hr)
    k = min(CROP, g.shape[-2], g.shape[-1])
    energy = F.avg_pool2d(g, k, stride=8)
    saturated = F.avg_pool2d(((hr < 0.02) | (hr > 0.98)).float()[..., 1:-1, 1:-1], k, stride=8)
    limit = max(0.01, float(saturated.min()) + 0.05)   # least-saturated windows when none is clean
    energy = energy.masked_fill(saturated > limit, -1.0)
    iy, ix = divmod(int(energy.flatten().argmax()), energy.shape[-1])
    return min(iy * 8 + 1, hr.shape[-2] - k), min(ix * 8 + 1, hr.shape[-1] - k), k


cases = {f"chosen: {i}": i for i in VISUAL_IMAGES} if VISUAL_IMAGES else pick_cases(results)
fig, axes = plt.subplots(len(cases), 5, figsize=(17, 4.0 * len(cases)), squeeze=False)
for row_axes, (why, image_id) in zip(axes, cases.items()):
    hr, outs = process(image_id)
    y, x, k = detail_crop(hr)
    r = results.set_index("image").loc[image_id]
    row_axes[0].imshow(hr[0, 0], cmap="gray", vmin=0, vmax=1)
    row_axes[0].add_patch(Rectangle((x, y), k, k, fill=False, ec="#FFD400", lw=1.5))
    row_axes[0].set_title(f"{image_id}\n{why}", fontsize=9)
    panels = [("HR", hr, "")] + [(m, outs[m], f"\nPSNR {r[f'{m}_psnr']:.2f}  SSIM {r[f'{m}_ssim']:.3f}"
                                             + ("" if np.isnan(r[f"{m}_lpips"]) else f"  LPIPS {r[f'{m}_lpips']:.3f}"))
                                  for m in METHODS]
    for ax, (name, img, label) in zip(row_axes[1:], panels):
        ax.imshow(img[0, 0, y:y + k, x:x + k], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(name + label, fontsize=9)
    for ax in row_axes:
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "visual_comparison.png", dpi=110)
plt.show()

## Image gallery: C8 vs C16 on unseen images
One large figure per image, so the two widths can be compared by eye. The images are validation images
(never trained on), spread over the whole range from hardest to easiest for bicubic, so the gallery is not
cherry-picked. Top row: the full image — LR input (enlarged ×4 without smoothing), bicubic, C8, C16 and
HR — with the zoom box; bottom row: the same zoomed region of each. Same gray range (0–1) everywhere.
The full-size outputs are also saved as PNG files (`gallery/`) and packed into `gallery.zip`.
`GALLERY_IMAGES` sets how many; `VISUAL_IMAGES` (above) overrides the choice.

In [ ]:
import shutil

from PIL import Image

GALLERY_IMAGES = 8
GALLERY_DIR = OUTPUT_DIR / "gallery"
GALLERY_DIR.mkdir(parents=True, exist_ok=True)


def to_png(img, path):
    Image.fromarray((img[0, 0].clamp(0, 1).numpy() * 255).round().astype(np.uint8)).save(path)


if VISUAL_IMAGES:
    gallery = list(VISUAL_IMAGES)
else:
    order = results.sort_values("bicubic_psnr")["image"].tolist()     # hardest -> easiest for bicubic
    gallery = list(dict.fromkeys(order[round(q * (len(order) - 1))] for q in np.linspace(0, 1, GALLERY_IMAGES)))

by_id = results.set_index("image")
for n, image_id in enumerate(gallery, 1):
    hr, outs = process(image_id)
    lr = degrade(hr[0], filename=image_id, scale=SCALE).unsqueeze(0)
    lr_big = F.interpolate(lr, scale_factor=SCALE, mode="nearest")
    y, x, k = detail_crop(hr)
    r = by_id.loc[image_id]
    caption = lambda m: (f"PSNR {r[f'{m}_psnr']:.2f}  SSIM {r[f'{m}_ssim']:.3f}"
                         + ("" if np.isnan(r[f"{m}_lpips"]) else f"\\nLPIPS {r[f'{m}_lpips']:.3f}")
                         + f"  sharp {r[f'{m}_sharpness']:.2f}×")
    panels = [("LR input (×4, no smoothing)", lr_big, "")] + [(m, outs[m], caption(m)) for m in METHODS] + [("HR (ground truth)", hr, "")]
    fig, axes = plt.subplots(2, len(panels), figsize=(3.6 * len(panels), 8.2))
    for col, (name, img, cap) in enumerate(panels):
        axes[0, col].imshow(img[0, 0], cmap="gray", vmin=0, vmax=1)
        axes[0, col].add_patch(Rectangle((x, y), k, k, fill=False, ec="#FFD400", lw=1.4))
        axes[0, col].set_title(name + (f"\\n{cap}" if cap else ""), fontsize=9.5)
        axes[1, col].imshow(img[0, 0, y:y + k, x:x + k], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        axes[1, col].set_title(f"{name} — zoom", fontsize=9.5)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"[{n}/{len(gallery)}] {image_id} — validation image (unseen)   |   "
                 f"C16 − C8: {r['delta_psnr']:+.2f} dB PSNR, {r['delta_ssim']:+.3f} SSIM"
                 + ("" if np.isnan(r["delta_lpips"]) else f", {r['delta_lpips']:+.3f} LPIPS"), fontsize=11)
    plt.tight_layout()
    plt.savefig(GALLERY_DIR / f"{n:02d}_{image_id}_compare.png", dpi=100)
    plt.show()
    for name, img in (("hr", hr), ("bicubic", outs["bicubic"]), *((m, outs[m]) for m in MODELS)):
        to_png(img, GALLERY_DIR / f"{n:02d}_{image_id}_{name}.png")

gallery_zip = shutil.make_archive(str(OUTPUT_DIR / "gallery"), "zip", GALLERY_DIR)
print(f"{len(gallery)} images; figures and full-size outputs in {GALLERY_DIR.name}/ and {Path(gallery_zip).name}")

## Save and download

In [ ]:
import platform

REPORT = {
    "status": "exploratory (validation split, one seed per width)",
    "code_branch": CODE_BRANCH,
    "code_commit": COMMIT,
    "split_seed": SPLIT_SEED,
    "n_val_images": len(results),
    "checkpoints": META,
    "same_training_iterations": SAME_BUDGET,
    "train_info": TRAIN_INFOS,
    "quality": summary.to_dict(orient="index"),
    "paired_C16_minus_C8": PAIRED,
    "visual_cases": cases,
    "gallery_images": gallery,
    "lpips_available": LPIPS_NET is not None,
    "cost": {k: {kk: (None if isinstance(vv, float) and np.isnan(vv) else vv) for kk, vv in v.items()}
             for k, v in COST.items()},
    "env": {"python": platform.python_version(), "torch": torch.__version__,
            "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"},
}
report_path = OUTPUT_DIR / "width_comparison_report.json"
report_path.write_text(json.dumps(REPORT, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("saved:", report_path.name, "val_results.csv", "width_comparison.png", "visual_comparison.png")
if IN_COLAB:
    from google.colab import files
    for f in (report_path, OUTPUT_DIR / "val_results.csv", OUTPUT_DIR / "width_comparison.png",
              OUTPUT_DIR / "visual_comparison.png", OUTPUT_DIR / "training_curves.png",
              OUTPUT_DIR / "gallery.zip"):
        if not f.exists():
            continue
        files.download(str(f))

## Conclusion

In [ ]:
p, s = PAIRED["PSNR (dB)"], PAIRED["SSIM"]
c8, c16 = COST["C8"], COST["C16"]
verdict = ("C16 is better" if p["ci95_low"] > 0 else
           "C8 is better" if p["ci95_high"] < 0 else
           "no clear PSNR difference")
print(f"Validation split, {len(results)} images, {META['C8']['iteration']} vs {META['C16']['iteration']} training iterations, split seed {SPLIT_SEED}.")
print(f"PSNR C16 − C8: {p['mean']:+.3f} dB (95% CI {p['ci95_low']:+.3f} … {p['ci95_high']:+.3f}); "
      f"C16 better on {p['C16_better']}/{p['n']} images  ->  {verdict} at this budget.")
print(f"SSIM C16 − C8: {s['mean']:+.4f} (95% CI {s['ci95_low']:+.4f} … {s['ci95_high']:+.4f}).")
lp = PAIRED["LPIPS (lower = better)"]
if lp["n"]:
    print(f"LPIPS C16 − C8: {lp['mean']:+.4f} (95% CI {lp['ci95_low']:+.4f} … {lp['ci95_high']:+.4f}; negative = C16 closer "
          f"perceptually); C16 better on {lp['C16_better']}/{lp['n']} images.")
for m in METHODS:
    line = f"{m:>8}: PSNR {results[f'{m}_psnr'].mean():.2f} dB, SSIM {results[f'{m}_ssim'].mean():.4f}"
    if lp["n"]:
        line += f", LPIPS {results[f'{m}_lpips'].mean():.4f}"
    line += f", edge sharpness {results[f'{m}_sharpness'].mean():.2f}× HR, edge error {results[f'{m}_edge_err'].mean():.4f}"
    print(line)
print(f"Cost C16 / C8: params ×{c16['params'] / c8['params']:.2f}, FLOPs ×{c16['gflops_per_tile'] / c8['gflops_per_tile']:.2f}, "
      f"latency ×{c16['latency_ms_per_batch'] / c8['latency_ms_per_batch']:.2f}"
      + (f", train time ×{c16['train_minutes'] / c8['train_minutes']:.2f}" if not np.isnan(c8['train_minutes']) else ""))

**Reading PSNR vs the eye:** PSNR/SSIM favour smooth outputs, so a model can lose to bicubic on PSNR
and still look sharper; LPIPS, edge sharpness/error and the visual panel show whether that is the case.
A large PSNR loss on smooth images (several dB) is more likely an artifact than "extra detail" — check it
in the visual panel.

**Limitations:** one training seed per width (seed-to-seed variation is not measured yet — train a second
seed with the same `SPLIT_SEED` before drawing conclusions); a fixed short budget (the wider model may need
more iterations to converge, so a small gap here may change with longer training); validation split only;
timings depend on the GPU and are only comparable within one session.